# D16 · IA generativa y LLMs

**Formación Pública — Bootcamp de Ciencia de Datos para funcionarios públicos**
Bloque avanzado · IA aplicada (opcional) · Semana 19

---

## ¿Qué vas a lograr?

En D15 le enseñaste a la máquina a **contar palabras**. Pero contar no es entender: tu clasificador no sabía que "jeringa" es algo médico, porque esa palabra no aparecía en el nombre del rubro. Hoy damos el salto: vas a usar un **modelo de lenguaje grande (LLM)** —la tecnología detrás de ChatGPT, Gemini o Claude— que **sí capta el significado**, los sinónimos y el contexto.

Vas a conectar tu cuaderno a un LLM **real y gratuito**, y a usarlo para tareas concretas de gobierno: explicar, clasificar y **extraer datos estructurados** de texto de compras públicas. Y —igual de importante— vas a aprender a usarlo **de forma responsable**: estos modelos se equivocan con seguridad y aprenden de lo que les mandas.

Al terminar serás capaz de:

- Entender, a nivel intuitivo, qué es un LLM y por qué "predecir la próxima palabra" produce algo tan útil.
- **Llamar a un LLM desde Python** y guiarlo con buenos *prompts*.
- Usarlo para **clasificar** y **extraer información estructurada** (JSON).
- Aplicar reglas básicas de **uso responsable** (privacidad, verificación).

**Competencia de salida:** resolver una tarea real de texto del Estado con un LLM, de forma reproducible y responsable.

**Entregable:** que las **4 celdas de chequeo** muestren ✅.

> **Funciona sin tarjeta y sin conexión.** Si configuras tu API key gratuita, el cuaderno habla con el modelo **en vivo**. Si no, usa respuestas reales **en caché** y los chequeos pasan igual. Nadie se queda fuera.


## 1. ¿Qué es un LLM (y por qué funciona)?

Un **LLM** (Large Language Model, modelo de lenguaje grande) es un programa entrenado con cantidades enormes de texto para una tarea engañosamente simple: **predecir la siguiente palabra**. Dale "El cielo es de color..." y aprende que "azul" es muy probable. Repite eso miles de millones de veces sobre casi todo el texto de internet y ocurre algo asombroso: para predecir bien, el modelo termina **capturando patrones del lenguaje y del mundo**. Por eso puede resumir, traducir, clasificar o responder, aunque nadie lo programó explícitamente para cada tarea.

La diferencia con D15 es enorme. Antes **contábamos** palabras; el modelo no entendía nada. Ahora el LLM **relaciona significados**: sabe que "jeringa", "guante quirúrgico" e "insumo clínico" pertenecen al mundo médico, aunque las palabras no coincidan. Eso es lo que lo hace tan potente para texto del Estado, que está lleno de sinónimos y jerga.

### El prompt: tú diriges

La instrucción que le das al modelo se llama **prompt**. Es tu herramienta de control, y escribir buenos prompts es una habilidad. Tres reglas que rinden de inmediato:

1. **Sé específico:** di exactamente qué quieres y en qué formato ("responde solo con el nombre del rubro", "devuelve un JSON con estas claves").
2. **Da contexto:** pega la lista de opciones, el ejemplo, el texto a analizar.
3. **Acota la salida:** si quieres una palabra, pídela; si quieres JSON, dilo y prohíbe el texto extra.

### Es probabilístico (y eso cambia cómo lo evaluamos)

Un LLM **no es determinista**: el mismo prompt puede dar respuestas algo distintas cada vez. Por eso, cuando programamos con LLMs, **no verificamos texto exacto**, sino la **estructura** de la respuesta: ¿es un JSON válido?, ¿la categoría está dentro de las permitidas?, ¿el resumen es más corto que el original? Verás ese estilo en los chequeos de hoy.

### Uso responsable (esto no es opcional en el Estado)

- **Alucinaciones:** el modelo puede inventar datos con total seguridad (cifras, leyes, nombres). **Nunca** publiques un número que dé un LLM sin verificarlo contra la fuente oficial.
- **Privacidad:** la capa **gratuita** de muchos proveedores **usa tus textos para entrenar**. No envíes datos personales (RUT, nombres, datos de salud) ni información reservada.
- **El responsable eres tú:** el LLM asiste; la decisión y la responsabilidad pública siguen siendo humanas.


## 2. Tu API key gratuita (Google AI Studio)

Usaremos **Gemini** (modelo `gemini-2.5-flash`) a través de **Google AI Studio**. Es la opción ideal para este bootcamp: **gratis, sin tarjeta de crédito y solo con tu cuenta de Google** (la misma de Colab).

**Cómo obtenerla (2 minutos):**

1. Entra a **https://aistudio.google.com/apikey** e inicia sesión con tu cuenta de Google.
2. Pulsa **"Create API key"** y copia la clave.
3. En Colab, ábrela de forma segura: panel izquierdo → 🔑 **Secrets** → **Add new secret** → nombre `GEMINI_API_KEY`, pega el valor y activa el acceso para el notebook.

> 🔒 **Nunca** escribas tu API key directamente en una celda (quedaría guardada en el archivo). Usa siempre los *Secrets* de Colab.

Si no quieres configurarla ahora, **no pasa nada**: el cuaderno funciona en **modo caché** y todos los ejercicios se pueden completar igual.

> **Sobre la capa gratuita:** permite del orden de 1.500 solicitudes al día sin costo. Ten presente que, en el plan gratuito, Google puede usar tus textos para mejorar sus modelos: por eso practicamos con datos públicos, nunca personales.


In [ ]:
# Instala el SDK oficial de Google (en Colab; si ya está, no hace nada)
!pip install -q google-genai

import os, json, re, unicodedata

MODELO = "gemini-2.5-flash"   # modelo gratuito de la capa sin tarjeta

# ---- Carga de la API key (Colab Secrets → variable de entorno → None) ----
def _cargar_api_key():
    try:
        from google.colab import userdata
        k = userdata.get("GEMINI_API_KEY")
        if k:
            return k
    except Exception:
        pass
    return os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")

API_KEY = _cargar_api_key()

# ---- Respuestas reales en caché (para funcionar sin red ni key) ----
CACHE = {
    "generico": ("Una licitación pública es el proceso formal y transparente con el que el "
                 "Estado invita a empresas a ofertar para proveer un bien o servicio y elige "
                 "la mejor oferta según criterios objetivos."),
    "clasificar": "Equipamiento y suministros médicos",
    "extraer": ('{"objeto": "Adquisición de productos de aseo e higiene", '
                '"categoria": "Equipos y suministros de limpieza", "es_servicio": false}'),
}

def _llamar_gemini(prompt):
    from google import genai
    client = genai.Client(api_key=API_KEY)
    resp = client.models.generate_content(model=MODELO, contents=prompt)
    return resp.text or ""

def preguntar_llm(prompt, tarea="generico"):
    # Patrón 'en vivo o caché': usa la API real si hay key; si no, cae a la caché.
    if API_KEY:
        try:
            return _llamar_gemini(prompt)
        except Exception as e:
            print(f"⚠️ La llamada en vivo falló ({e}). Uso la respuesta en caché.")
    return CACHE.get(tarea, CACHE["generico"])

# Auxiliar para los chequeos: minúsculas sin tildes
def _sa(s):
    s = unicodedata.normalize("NFKD", str(s).lower())
    return "".join(c for c in s if not unicodedata.combining(c))

if API_KEY:
    print("🔑 API key detectada → modo EN VIVO con", MODELO)
else:
    print("ℹ️ Sin API key → modo CACHÉ (offline). Los ejercicios funcionan igual.")

### Prueba de humo

Ejecuta esto para ver el patrón funcionando. En modo en vivo verás una respuesta del modelo; en modo caché, la respuesta guardada.


In [ ]:
print(preguntar_llm("Explica en una frase qué es una licitación pública.", tarea="generico"))

## Ejercicio 1 — Tu primer prompt

Vas a pedirle al modelo una explicación clara. Lo importante aquí es **el prompt**: una buena instrucción produce una buena respuesta.

Tu tarea:

1. Escribe en `prompt1` una instrucción que pida explicar, **en una sola frase y en lenguaje simple**, qué es una **licitación pública**.
2. La línea provista llama al modelo y guarda la respuesta en `respuesta1`.

> 💡 **Pista:** sé específico con el formato. Por ejemplo: *"Explica en una sola frase, en lenguaje simple para un funcionario, qué es una licitación pública."*

> ⚠️ **Error típico:** prompts vagos como "licitación". Mientras más claro el pedido (tema + formato + audiencia), mejor la respuesta.


In [ ]:
# TODO: escribe un buen prompt pidiendo una explicación en una frase simple
prompt1 = None   # <-- reemplaza por tu prompt

respuesta1 = preguntar_llm(prompt1, tarea="generico")
print(respuesta1)

In [ ]:
# Celda de chequeo — Ejercicio 1
def _chequeo_1():
    try:
        p, r = prompt1, respuesta1
    except NameError:
        print("❌ Faltan `prompt1` o `respuesta1`. Completa el TODO."); return
    if not p:
        print("❌ `prompt1` está vacío. Escribe una instrucción clara."); return
    try:
        assert isinstance(p, str) and len(p) >= 15, "El prompt debería ser una instrucción con sentido (15+ caracteres)."
        assert "licitaci" in _sa(p), "El prompt debería pedir explícitamente qué es una *licitación* pública."
        assert isinstance(r, str) and len(r.strip()) > 0, "La respuesta del modelo llegó vacía."
        print("✅ Correcto: enviaste un prompt claro y recibiste respuesta del modelo.")
        print("   Respuesta:", r.strip()[:120], "...")
    except AssertionError as e:
        print(f"❌ Casi. Pista: {e}")

_chequeo_1()

## Ejercicio 2 — Clasificar *con comprensión* (lo que D15 no podía)

Recuerda el límite de D15: una solicitud que decía "jeringas y guantes para el hospital" **no** se podía clasificar por palabras clave, porque esas palabras no aparecen en el nombre del rubro. El LLM **sí** lo resuelve, porque entiende que eso es del mundo médico.

Programa la función `clasificar_llm(texto)` que:

1. Construya un prompt que **incluya la lista `RUBROS`** y pida elegir el rubro más adecuado para `texto`.
2. Pida responder **solo con el nombre exacto del rubro** (sin explicaciones).
3. Llame a `preguntar_llm(prompt, tarea="clasificar")` y devuelva el resultado **sin espacios sobrantes** (`.strip()`).

> 💡 **Pista para armar el prompt:**
> ```python
> opciones = "\n".join("- " + r for r in RUBROS)
> prompt = f"Eres un asistente de compras públicas. Elige el rubro MÁS adecuado:\n{opciones}\nSolicitud: \"{texto}\"\nResponde SOLO con el nombre exacto del rubro."
> ```

> ⚠️ **Recuerda:** como el modelo es probabilístico, el chequeo no exige texto exacto, sino que la categoría sea la correcta (médica).


In [ ]:
# Lista de rubros reales (subconjunto del catálogo de ChileCompra) para clasificar
RUBROS = [
    "Alimentos, bebidas y tabaco",
    "Artículos de electrónica",
    "Combustibles, lubricantes y anticorrosivos",
    "Educación, formación, entrenamiento y capacitación",
    "Equipamiento y suministros médicos",
    "Equipos y suministros de limpieza",
    "Equipos, accesorios y suministros de oficina",
    "Maquinaria para construcción y edificación",
    "Medicamentos y productos farmacéuticos",
    "Muebles y mobiliario",
    "Servicios de construcción y mantenimiento",
    "Servicios de limpieza industrial",
]
print(f"{len(RUBROS)} rubros disponibles para clasificar.")

In [ ]:
def clasificar_llm(texto):
    # TODO: arma un prompt con RUBROS, pide elegir uno y devuélvelo sin espacios sobrantes
    return None   # <-- reemplaza por tu implementación

# Prueba con un caso que el método de palabras clave de D15 no podía resolver:
print(clasificar_llm("Necesitamos jeringas, guantes de látex y mascarillas para el hospital."))

In [ ]:
# Celda de chequeo — Ejercicio 2
def _chequeo_2():
    try:
        f = clasificar_llm
    except NameError:
        print("❌ Aún no defines `clasificar_llm`."); return
    try:
        r = f("Necesitamos jeringas, guantes de látex y mascarillas para el hospital.")
        assert r is not None, "La función devuelve None. Completa la implementación."
        assert isinstance(r, str) and len(r) > 0, "Debe devolver el nombre del rubro como texto."
        assert "medic" in _sa(r), \
            f"Para insumos de hospital se esperaba un rubro médico, y devolvió «{r}»."
        print("✅ Correcto: el LLM clasificó por *significado*, no por palabras en común.")
        print("   «jeringas, guantes, mascarillas…» →", r)
    except AssertionError as e:
        print(f"❌ Casi. Pista: {e}")
    except Exception as e:
        print(f"❌ La función falló: {e}. Revisa el prompt y la llamada a preguntar_llm.")

_chequeo_2()

## Ejercicio 3 — Extraer datos estructurados (JSON)

Una de las cosas más útiles de un LLM en el Estado: convertir **texto libre** en **datos ordenados**. Le pasas la descripción de una licitación y te devuelve campos limpios que podrías guardar en una tabla.

Programa la función `extraer_json(texto)` que:

1. Pida al modelo un **JSON** con exactamente estas claves:
   - `"objeto"`: de qué trata, en pocas palabras (texto).
   - `"categoria"`: el rubro general (texto).
   - `"es_servicio"`: `true` si es un servicio, `false` si es un bien (booleano).
2. Pida responder **solo el JSON** (sin explicaciones ni ```).
3. Llame a `preguntar_llm(prompt, tarea="extraer")`, **limpie** posibles ``` y convierta el texto a diccionario con `json.loads`.

> 💡 **Pista para limpiar y parsear:**
> ```python
> bruto = preguntar_llm(prompt, tarea="extraer").strip()
> bruto = bruto.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
> return json.loads(bruto)
> ```

> ⚠️ **Error típico:** olvidar pedir "solo el JSON". Si el modelo agrega texto antes o después, `json.loads` falla. Un buen prompt evita el problema.


In [ ]:
def extraer_json(texto):
    # TODO: pide un JSON con objeto/categoria/es_servicio, limpia y parsea con json.loads
    return None   # <-- reemplaza por tu implementación

ejemplo = "Convenio Marco para la adquisición de productos de aseo e higiene. Producto: toallas de papel."
print(extraer_json(ejemplo))

In [ ]:
# Celda de chequeo — Ejercicio 3
def _chequeo_3():
    try:
        f = extraer_json
    except NameError:
        print("❌ Aún no defines `extraer_json`."); return
    try:
        d = f("Convenio Marco para la adquisición de productos de aseo e higiene. Producto: toallas de papel.")
        assert d is not None, "La función devuelve None. Completa la implementación."
        assert isinstance(d, dict), "El resultado debe ser un diccionario (usa json.loads)."
        for k in ("objeto", "categoria", "es_servicio"):
            assert k in d, f"Falta la clave '{k}' en el JSON."
        assert isinstance(d["es_servicio"], bool), "'es_servicio' debe ser booleano (true/false), no texto."
        assert isinstance(d["objeto"], str) and len(d["objeto"]) > 0, "'objeto' debe ser texto no vacío."
        print("✅ Correcto: texto libre → datos estructurados.")
        print("   JSON:", d)
    except AssertionError as e:
        print(f"❌ Casi. Pista: {e}")
    except Exception as e:
        print(f"❌ Falló el parseo: {e}. ¿Pediste 'solo el JSON' y limpiaste los ```?")

_chequeo_3()

## Ejercicio 4 — Uso responsable: no entregues datos personales

La capa gratuita del modelo **puede usar tus textos para entrenar**. Por eso, antes de enviar cualquier texto, conviene revisar que **no contenga datos personales**. El caso más típico en Chile: un **RUT**.

Programa un **guardrail** (una barrera de seguridad): la función `es_seguro_enviar(texto)` que devuelva `False` si el texto **contiene un RUT**, y `True` si no.

Un RUT chileno tiene la forma `12.345.678-9` (puntos opcionales, dígito verificador que puede ser un número o la letra K). Esta expresión regular lo detecta:

```python
patron_rut = r"\b\d{1,2}\.?\d{3}\.?\d{3}-[\dkK]\b"
```

Tu tarea: usa `re.search` con ese patrón y devuelve `True` solo si **no** hay coincidencia.

> 💡 **Pista:** `re.search(patron, texto) is None` es `True` cuando el texto está limpio.

> ⚠️ **Idea de fondo:** un guardrail no reemplaza el criterio, pero evita el error más común: mandar sin querer datos sensibles a un servicio externo. En el Estado, esto importa de verdad.


In [ ]:
def es_seguro_enviar(texto):
    # TODO: devuelve False si el texto contiene un RUT; True si está limpio
    patron_rut = r"\b\d{1,2}\.?\d{3}\.?\d{3}-[\dkK]\b"
    return None   # <-- reemplaza por tu implementación

print(es_seguro_enviar("Comprar 500 resmas de papel tamaño carta"))
print(es_seguro_enviar("Proveedor Juan Pérez, RUT 12.345.678-9, entrega el martes"))

In [ ]:
# Celda de chequeo — Ejercicio 4
def _chequeo_4():
    try:
        f = es_seguro_enviar
    except NameError:
        print("❌ Aún no defines `es_seguro_enviar`."); return
    try:
        limpio = f("Comprar 500 resmas de papel tamaño carta")
        con_rut = f("Proveedor Juan Pérez, RUT 12.345.678-9, entrega el martes")
        assert limpio is not None and con_rut is not None, "La función devuelve None. Complétala."
        assert limpio is True, "Un texto sin datos personales debería ser seguro (True)."
        assert con_rut is False, "Un texto con RUT NO debería ser seguro (False)."
        assert f("RUT 9.876.543-K en la oferta") is False, "También debe detectar el dígito verificador 'K'."
        print("✅ Correcto: el guardrail bloquea textos con datos personales antes de enviarlos.")
    except AssertionError as e:
        print(f"❌ Casi. Pista: {e}")
    except Exception as e:
        print(f"❌ La función falló: {e}.")

_chequeo_4()

## Cierre — Lo que te llevas

- Un **LLM** aprende a predecir la próxima palabra y, de paso, capta **significado**: por eso entiende sinónimos y contexto que el conteo de palabras (D15) no podía.
- Tú lo diriges con el **prompt**: específico, con contexto y con el formato de salida acotado.
- Es **probabilístico**: validamos la **estructura** de la respuesta, no el texto exacto.
- Sirve para tareas reales del Estado: **clasificar** por significado y **extraer** datos estructurados (JSON) de texto libre.
- **Uso responsable, siempre:** verifica las cifras contra la fuente, no envíes datos personales, y recuerda que la responsabilidad pública es humana.

### Hacia dónde sigue
Hoy el modelo respondió solo con lo que "sabe" de su entrenamiento. Pero el Estado tiene su **propia** información (normativa, fichas, bases) que el modelo no vio. En **D17 (RAG)** aprenderás a **darle al LLM tus documentos** para que responda con datos tuyos, citando la fuente. Y en **D18**, a encadenar todo esto en **agentes**.

---
*Formación Pública · Contenido bajo licencia CC BY 4.0. LLM: Google Gemini (capa gratuita). Datos: ChileCompra / MercadoPúblico.*
